# Phân Loại Tiếng Nói (Speech) Và Âm Nhạc (Music) Sử Dụng Đặc Trưng Thính Giác (Auditory Features) & SVM

Jupyter Notebook này được xây dựng dựa trên bài tập thực hành của **Slide 13 & 14** thuộc Bài 4: **Hearing, Auditory Models, and Speech Perception**.

### Mục tiêu bài tập:
1. **Mô phỏng thính giác người**: Trích xuất đặc trưng phổ **MFCC** (Mel-Frequency Cepstral Coefficients) từ tín hiệu âm thanh.
2. **Tiền xử lý dữ liệu**: Xử lý tập tin âm thanh từ thư mục người dùng nhập vào, chuẩn hóa đặc trưng bằng `StandardScaler`.
3. **Phân loại bằng Machine Learning**: Xây dựng và huấn luyện mô hình Support Vector Machine (**SVM**) với kernel tuyến tính để phân biệt hai lớp âm thanh: **Tiếng nói (Speech)** và **Âm nhạc (Music)**.
4. **Đánh giá hiệu năng**: Tính toán độ chính xác (**Accuracy score**) trên tập kiểm thử.

In [ ]:
# ========================================== 
# 1. IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ==========================================
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# Thư viện học máy scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Đã import tất cả các thư viện thành công!")

--- 
## Bước 2: Cấu Hình Thư Mục Dataset Đầu Vào (Dataset Input Configuration)

Hãy nhập đường dẫn tới thư mục chứa các tệp âm thanh thực tế của bạn vào ô dưới đây.
- Bạn có thể sắp xếp dữ liệu bằng cách đặt các tệp `.wav` vào chung một thư mục chính và đặt tên có chứa từ khóa **`speech`** hoặc **`music`** (ví dụ: `speech_01.wav`, `guitar_music.wav`).
- Hoặc chia thành 2 thư mục con riêng biệt là `/speech` và `/music` bên trong thư mục chính.

> **Lưu ý quan trọng**: Bộ dữ liệu đầu vào bắt buộc phải có **cả hai lớp** tệp tin (Tiếng nói và Âm nhạc) thì mô hình SVM mới có thể huấn luyện phân loại được.

In [ ]:
# ==========================================
# 2. ĐƯỜNG DẪN ĐẾN THƯ MỤC DATASET CỦA BẠN
# ==========================================
# Hãy sửa đường dẫn dưới đây thành thư mục chứa các file .wav thực tế của bạn
DATASET_DIR = "c:/Users/PC/OneDrive/Desktop/SPL301/practice_analysis/"

print(f"Thư mục dataset đầu vào đã được cấu hình: '{DATASET_DIR}'")

--- 
## Bước 3: Định Nghĩa Các Hàm Trích Xuất Đặc Trưng (Feature Extraction)

Theo như hướng dẫn giải trong slide:
1. Hàm `extract_features(file_path)`: Nạp file âm thanh, trích xuất 13 đặc trưng MFCC cơ bản, sau đó tính **giá trị trung bình cộng (Mean)** dọc theo trục thời gian (`axis=1`). Việc lấy trung bình này giúp biến đổi dữ liệu từ dạng ma trận 2D thành một véc-tơ đặc trưng 1D có kích thước cố định (13 phần tử), phù hợp làm đầu vào cho mô hình SVM.
2. Hàm `get_label_from_filename(file_path)`: Trích xuất nhãn phân loại từ tên tệp tin hoặc tên thư mục cha.
   - Nếu tên tệp hoặc thư mục cha chứa chữ `speech` $\rightarrow$ Nhãn là `0` (Speech).
   - Nếu tên tệp hoặc thư mục cha chứa chữ `music` $\rightarrow$ Nhãn là `1` (Music).
3. Hàm `preprocess_dataset(file_paths)`: Duyệt qua tất cả đường dẫn tệp tin, trích xuất đặc trưng và lưu nhãn tương ứng.

In [ ]:
# ==========================================
# 3. ĐỊNH NGHĨA HÀM TRÍCH XUẤT ĐẶC TRƯNG & NHÃN
# ==========================================

# Hàm trích xuất đặc trưng MFCC từ file âm thanh
def extract_features(file_path):
    # Tải file âm thanh với độ dài tối đa 3 giây
    y, sr = librosa.load(file_path, duration=3) 
    
    # Trích xuất 13 đặc trưng MFCC
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) 
    
    # Tính giá trị trung bình dọc theo trục thời gian (axis=1)
    mfccs_mean = np.mean(mfccs, axis=1) 
    
    return mfccs_mean

# Hàm lấy nhãn tự động từ tên file hoặc đường dẫn (Speech = 0, Music = 1)
def get_label_from_filename(file_path):
    # Chuyển đường dẫn thành chữ thường để tìm kiếm không phân biệt hoa thường
    path_lower = file_path.lower()
    
    # Kiểm tra từ khóa trong tên file hoặc thư mục cha chứa nó
    if 'speech' in path_lower:
        return 0  # 0 đại diện cho Tiếng nói
    elif 'music' in path_lower:
        return 1  # 1 đại diện cho Âm nhạc
    else:
        # Nếu không có từ khóa nào, báo lỗi cụ thể để cấu hình lại
        raise ValueError(f"Không thể xác định nhãn cho file: '{os.path.basename(file_path)}'. Tên file hoặc đường dẫn của nó phải chứa từ khóa 'speech' hoặc 'music'.")

# Hàm tiền xử lý tập dữ liệu âm thanh và trích xuất đặc trưng
def preprocess_dataset(file_paths):
    features = []
    labels = []
    
    for path in file_paths:
        try:
            # Trích xuất đặc trưng trung bình và thêm vào list
            feat = extract_features(path)
            lbl = get_label_from_filename(path)
            
            features.append(feat)
            labels.append(lbl)
        except Exception as e:
            print(f"Bỏ qua file '{os.path.basename(path)}' do gặp lỗi: {e}")
        
    return np.array(features), np.array(labels)

--- 
## Bước 4: Chuẩn Bị Bộ Dữ Liệu Học Máy (Data Preparation)

Chúng ta sẽ quét qua thư mục đã được cấu hình ở Bước 2 để tìm toàn bộ các tệp tin có đuôi `.wav` (bao gồm cả trong các thư mục con nếu có), sau đó thực hiện trích xuất đặc trưng.

In [ ]:
# ==========================================
# 4. TIẾN HÀNH QUÉT VÀ TIỀN XỬ LÝ DỮ LIỆU ĐẦU VÀO
# ==========================================
# Tìm kiếm tất cả tệp .wav trong thư mục cấu hình và các thư mục con của nó
dataset_paths = glob.glob(os.path.join(DATASET_DIR, "**", "*.wav"), recursive=True)

print(f"Tìm thấy {len(dataset_paths)} tệp âm thanh .wav trong thư mục được cấu hình.")

if len(dataset_paths) == 0:
    print("LỖI: Không tìm thấy tệp .wav nào! Vui lòng kiểm tra lại đường dẫn DATASET_DIR ở Bước 2.")
else:
    # Trích xuất đặc trưng và nhãn
    X, y = preprocess_dataset(dataset_paths)
    
    # Kiểm tra số lượng lớp dữ liệu
    unique_classes = np.unique(y) if len(y) > 0 else []
    
    print("\n--- THÔNG TIN DỮ LIỆU SAU TIỀN XỬ LÝ ---")
    print(f"Tổng số mẫu trích xuất thành công: {len(y)}")
    if len(y) > 0:
        print(f"Kích thước ma trận đặc trưng X (Features): {X.shape}  -> (Số mẫu, Số chiều đặc trưng)")
        print(f"Kích thước véc-tơ nhãn y (Labels):       {y.shape}  -> (Số mẫu,)")
        print(f"Số lượng mẫu thuộc lớp Speech (Nhãn 0): {np.sum(y == 0)}")
        print(f"Số lượng mẫu thuộc lớp Music  (Nhãn 1): {np.sum(y == 1)}")
        
        if len(unique_classes) < 2:
            print("\nCẢNH BÁO: Tập dữ liệu của bạn chỉ có 1 lớp duy nhất! Mô hình SVM yêu cầu ít nhất 2 lớp (Speech và Music) để huấn luyện phân loại.")

--- 
## Bước 5: Trực Quan Hóa Đồ Thị Đặc Trưng (Trực Quan Học Tập)

Hãy cùng vẽ đồ thị Waveform và phổ đặc trưng MFCC của mẫu Speech và mẫu Music đầu tiên tìm thấy trong tập dữ liệu của bạn để thấy sự khác biệt âm học rõ nét.

In [ ]:
# ==========================================
# 5. TRỰC QUAN HÓA DỮ LIỆU THỰC TẾ
# ==========================================
speech_files = [p for p in dataset_paths if 'speech' in p.lower()]
music_files = [p for p in dataset_paths if 'music' in p.lower()]

if len(speech_files) > 0 and len(music_files) > 0:
    # Load 1 tệp Speech và 1 tệp Music đại diện
    y_s, sr_s = librosa.load(speech_files[0], duration=3)
    y_m, sr_m = librosa.load(music_files[0], duration=3)
    
    plt.figure(figsize=(15, 8))
    
    # Vẽ Waveform của Speech
    plt.subplot(2, 2, 1)
    librosa.display.waveshow(y_s, sr=sr_s, color='blue')
    plt.title(f'Waveform: {os.path.basename(speech_files[0])} (Speech)')
    plt.xlabel('Thời gian (giây)')
    plt.ylabel('Biên độ')
    
    # Vẽ Waveform của Music
    plt.subplot(2, 2, 2)
    librosa.display.waveshow(y_m, sr=sr_m, color='orange')
    plt.title(f'Waveform: {os.path.basename(music_files[0])} (Music)')
    plt.xlabel('Thời gian (giây)')
    
    # Vẽ MFCCs của Speech
    plt.subplot(2, 2, 3)
    mfccs_s = librosa.feature.mfcc(y=y_s, sr=sr_s, n_mfcc=13)
    librosa.display.specshow(mfccs_s, x_axis='time')
    plt.colorbar()
    plt.title('Đặc Trưng MFCCs - Speech')
    plt.ylabel('Chỉ số MFCC')
    
    # Vẽ MFCCs của Music
    plt.subplot(2, 2, 4)
    mfccs_m = librosa.feature.mfcc(y=y_m, sr=sr_m, n_mfcc=13)
    librosa.display.specshow(mfccs_m, x_axis='time')
    plt.colorbar()
    plt.title('Đặc Trưng MFCCs - Music')
    
    plt.tight_layout()
    plt.show()
else:
    print("Bỏ qua bước vẽ đồ thị vì không tìm đủ cả 2 lớp Speech và Music trong tập dữ liệu của bạn để hiển thị đối chiếu.")

--- 
## Bước 6: Huấn Luyện & Đánh Giá Mô Hình SVM (Model Training & Evaluation)

Theo đúng mã nguồn giải trong slide:
1. **Chia tập dữ liệu (Train/Test Split)**: Chia 80% dữ liệu để huấn luyện (Train) và 20% để kiểm thử (Test) bằng hàm `train_test_split`.
2. **Chuẩn hóa thang đo (StandardScaler)**: Các đặc trưng MFCC có dải biên độ lớn nhỏ rất khác nhau (ví dụ hệ số MFCC thứ nhất thường có giá trị âm lớn, trong khi các hệ số sau rất nhỏ). Hàm `StandardScaler` sẽ chuẩn hóa đưa các giá trị về dạng phân phối chuẩn có $\mu = 0$ và $\sigma = 1$, giúp mô hình SVM hội tụ nhanh và đạt độ chính xác cao.
3. **Phân loại bằng SVM (Support Vector Classifier)**: Sử dụng mô hình `SVC` với kernel tuyến tính (`kernel='linear'`) và tham số điều hòa $C=1.0$.
4. **Đánh giá**: Dự đoán trên tập kiểm thử và xuất điểm số độ chính xác (**Accuracy Score**).

In [ ]:
# ==========================================
# 6. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH SVM
# ==========================================
def train_evaluate_model(features, labels):
    # Kiểm tra số lượng nhãn độc bản
    if len(np.unique(labels)) < 2:
        return "Không thể huấn luyện vì dữ liệu chỉ chứa 1 nhãn duy nhất! Bạn cần bổ sung thêm file của lớp còn lại."
    
    # Chia tập dữ liệu thành Train (80%) và Test (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    # Khởi tạo bộ chuẩn hóa thang đo
    scaler = StandardScaler()
    
    # Chuẩn hóa tập huấn luyện và áp dụng lên tập kiểm thử
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Khởi tạo mô hình phân loại SVM tuyến tính
    clf = SVC(kernel='linear', C=1.0, random_state=42)
    
    # Huấn luyện mô hình
    clf.fit(X_train_scaled, y_train)
    
    # Dự đoán trên tập kiểm thử
    y_pred = clf.predict(X_test_scaled)
    
    # Tính độ chính xác
    accuracy = accuracy_score(y_test, y_pred)
    
    # In các đánh giá chi tiết bổ sung
    print("\n--- BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT) ---")
    print(classification_report(y_test, y_pred, target_names=['Speech (0)', 'Music (1)']))
    
    print("--- MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ---")
    print(confusion_matrix(y_test, y_pred))
    
    return accuracy

# Chạy hàm huấn luyện và đánh giá
if 'y' in locals() and len(y) > 0:
    result = train_evaluate_model(X, y)
    if isinstance(result, float):
        print(f"\nKết quả đánh giá chung trên tập Test: Accuracy = {result * 100:.2f}%")
    else:
        print(f"\nThông báo: {result}")
else:
    print("Chưa có dữ liệu hợp lệ để thực hiện huấn luyện mô hình.")